# 1. Configuración del Entorno y Carga de Componentes

## 1.1. Importación de Librerías

Importamos las librerías necesarias para el análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import scanpy as sc
import joblib

# Configuramos el estilo de las visualizaciones
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 1.2. Definición de Rutas

Definimos las rutas a los datos de entrada y a las carpetas de salida.

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
MODELS_PATH = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'

# Nombres de los ficheros de entrada
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
REF_SC_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
MODEL_FILENAME = 'mlp_marker_genes.joblib'

os.makedirs(os.path.join(FIGURES_PATH, 'deconvolution'), exist_ok=True)

print("Rutas definidas.")

## 1.3. Carga de Datos de TCGA

Cargamos los datos de expresión y clínicos de la cohorte de TCGA-LUAD, que fueron procesados en el notebook anterior.

In [ ]:
print("Cargando datos de TCGA (bulk RNA-seq)...")
bulk_counts_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
bulk_clinical_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

print("Datos de TCGA cargados:")
print(f"  - Matriz de conteos: {bulk_counts_df.shape[0]} muestras x {bulk_counts_df.shape[1]} genes")
print(f"  - Datos clínicos: {bulk_clinical_df.shape[0]} muestras x {bulk_clinical_df.shape[1]} variables")


## 1.4. Carga de Datos de Referencia

Cargamos el objeto AnnData de scRNA-seq que contiene los perfiles de expresión de referencia para cada tipo celular.


In [ ]:
print("\nCargando datos de referencia (scRNA-seq)...")
adata_ref = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, REF_SC_FILENAME))

print("Datos de referencia cargados:")
print(adata_ref)

## 1.5. Carga del Modelo MLP seleccionado

Cargamos nuestro modelo MLP final, que fue entrenado y validado en el notebook anterior.

In [ ]:
print("\nCargando el modelo MLP entrenado...")
model_path = os.path.join(MODELS_PATH, MODEL_FILENAME)

# El modelo MLP se guardó como un diccionario con el modelo y el LabelEncoder
loaded_model_data = joblib.load(model_path)

# Verificamos si es un diccionario o el modelo directamente
if isinstance(loaded_model_data, dict):
    model = loaded_model_data['model']
    label_encoder = loaded_model_data.get('label_encoder', None)
else:
    model = loaded_model_data
    label_encoder = None

print("Modelo MLP cargado exitosamente.")
if label_encoder:
    print("LabelEncoder también cargado.")
    print("Clases del modelo:", label_encoder.classes_)
else:
    print("Clases del modelo:", model.classes_)

# Nos aseguramos de que los genes en los datos de bulk y de referencia son consistentes.
common_genes = list(set(bulk_counts_df.columns) & set(adata_ref.var.index))
print(f"\n[VERIFICACIÓN] Se encontraron {len(common_genes)} genes en común entre los datos de bulk y de referencia.")

try:
    assert len(common_genes) > 20000, "Hay pocos genes en común. Revisar la anotación de genes (Ensembl IDs)."
    print("[OK] El solapamiento de genes es alto")
except AssertionError as e:
    print(f"[ERROR] {e}")


Próximo, hacer la matriz de firmas y la deconvolución

# 2. Creación de la Matriz de Firmas Genéticas

## 2.1. Selección de los Genes Marcadores

El primer paso es definir qué genes formarán la base de nuestra firma. Para mantener la consistencia con nuestro modelo MLP optimizado, utilizaremos exactamente la misma lista de genes marcadores que se generó en el Notebook 2.

In [ ]:
print("--- Re-calculando la lista de genes marcadores desde los datos de referencia ---")

# Usamos rank_genes_groups en el objeto de referencia para obtener los marcadores
sc.tl.rank_genes_groups(adata_ref, groupby='cell_type', method='t-test', use_raw=False)

# Extraemos el DataFrame de marcadores
marker_genes_df = pd.DataFrame(adata_ref.uns['rank_genes_groups']['names'])

# Definimos cuántos genes por tipo celular
top_n_genes = 25 # Mismo valor que en el notebook anterior

# Creamos la lista final de Ensembl IDs
marker_genes_list = []
for col in marker_genes_df.columns:
    marker_genes_list.extend(marker_genes_df[col].head(top_n_genes))

# Eliminamos duplicados para tener la lista única de features
marker_genes_list = sorted(list(set(marker_genes_list)))

print(f"Se ha generado una lista de {len(marker_genes_list)} genes marcadores únicos.")

## 2.2. Cálculo de la Expresión Promedio por Tipo Celular

A continuación, calculamos la expresión promedio de cada gen marcador en cada tipo celular de nuestro dataset de referencia. Estos perfiles promedio formarán las "firmas" de cada población celular.


In [ ]:
print("\n--- Calculando perfiles de expresión promedio ---")

# Creamos un DataFrame a partir de la matriz de expresión log-normalizada de adata_ref
# Usamos .var_names para los nombres de las columnas (genes)
ref_expression_df = pd.DataFrame(
    adata_ref.X.toarray(),
    index=adata_ref.obs.index,
    columns=adata_ref.var.index
)

# Añadimos la columna 'cell_type' para poder agrupar
ref_expression_df['cell_type'] = adata_ref.obs['cell_type'].values

# Agrupamos por tipo celular y calculamos la media de la expresión de cada gen
signature_matrix = ref_expression_df.groupby('cell_type').mean()

# La matriz resultante tiene tipos celulares como filas y TODOS los genes como columnas.
# La transponemos y filtramos para quedarnos solo con nuestros genes marcadores.
signature_matrix = signature_matrix.T
signature_matrix = signature_matrix.loc[marker_genes_list]

print("Matriz de firmas genéticas creada con éxito.")
print(f"Dimensiones de la matriz de firmas: {signature_matrix.shape[0]} genes x {signature_matrix.shape[1]} tipos celulares")
display(signature_matrix.head())

## 2.3. Visualización de la Matriz de Firmas

Para verificar visualmente la calidad de nuestra matriz de firmas, la representamos como un heatmap. Esperamos ver patrones claros donde los genes marcadores muestran una alta expresión específica en su tipo celular correspondiente.

In [ ]:
print("\n--- Visualizando la matriz de firmas ---")

# Usamos un clustering jerárquico para agrupar genes y tipos celulares similares
# Esto a menudo revela bloques de co-expresión.
plt.figure(figsize=(12, 18))
sns.clustermap(
    signature_matrix,
    cmap='viridis',       
    standard_scale=0,     # Escalamos por gen (filas) para resaltar patrones relativos
    dendrogram_ratio=0.1
)
plt.suptitle('Heatmap Clusterizado de la Matriz de Firmas Genéticas', y=1.02)
plt.show()

# 3. Deconvolución de las Muestras de Bulk RNA-seq

## 3.1. Preparación y Armonización de Datos (Normalización a TPM)

Como se determinó en el análisis metodológico, es crucial que tanto la matriz de firmas como la de bulk estén en la misma escala. Implementamos una normalización a Transcripciones Por Millón (TPM), que corrige tanto por la profundidad de secuenciación como por la longitud de los genes, proporcionando una estimación más precisa de la abundancia de transcritos.

In [ ]:
from sklearn.svm import NuSVR
from tqdm.notebook import tqdm

print("Extrayendo la longitud de los genes desde los metadatos del objeto de referencia...")
gene_lengths = pd.to_numeric(adata_ref.raw.var['feature_length'])

print(f"Se han obtenido las longitudes para {len(gene_lengths)} genes desde la referencia scRNA-seq.")
print(f"La matriz de bulk contiene {bulk_counts_df.shape[1]} genes.")

# Encontramos el conjunto de genes para los que tenemos toda la información
common_genes_with_length = bulk_counts_df.columns.intersection(gene_lengths.index)

print(f"Se procederá con {len(common_genes_with_length)} genes que están presentes tanto en el bulk como en la referencia con longitud conocida.")

# Filtramos la matriz de bulk ANTES de la normalización
bulk_counts_aligned = bulk_counts_df[common_genes_with_length]
# Filtramos la serie de longitudes para que coincida exactamente
gene_lengths_aligned = gene_lengths[common_genes_with_length]

In [ ]:
from scipy.sparse import csr_matrix, diags
import numpy as np

def counts_to_tpm_sparse(sparse_counts_matrix, lengths_series):
    """
    Convierte una matriz de conteos dispersa (células x genes) a TPM.
    """
    # 1. Normalizar por longitud de gen en kilobases (RPK)
    lengths_kb = lengths_series.values / 1000
    inv_lengths_kb = 1 / lengths_kb
    rpk_matrix = sparse_counts_matrix.dot(diags(inv_lengths_kb))

    # 2. Calcular el "per million" scaling factor
    # Sumamos las filas para obtener el total de RPK por célula
    sum_rpk_per_cell = np.asarray(rpk_matrix.sum(axis=1))
    
    per_million_scalers = sum_rpk_per_cell / 1e6
    
    # 3. Dividir RPK por el factor de escala para obtener TPM
    # Evitamos la división por cero
    per_million_scalers[per_million_scalers == 0] = 1
    
    # Creamos una matriz diagonal inversa para la normalización final
    inv_scalers = 1 / per_million_scalers
    # .flatten() es importante para asegurar que inv_scalers es un vector 1D
    tpm_matrix = diags(inv_scalers.flatten()).dot(rpk_matrix)
    
    return tpm_matrix.tocsr()

In [ ]:
print("\nNormalizando matriz de bulk a TPM...")
# Alinear los genes como antes
common_genes_with_length = bulk_counts_df.columns.intersection(gene_lengths.index)
bulk_counts_aligned = bulk_counts_df[common_genes_with_length]
gene_lengths_aligned = gene_lengths[common_genes_with_length]
# Convertimos a matriz dispersa de scipy
bulk_counts_sparse = csr_matrix(bulk_counts_aligned.values)
# Aplicamos la nueva función
bulk_tpm_sparse = counts_to_tpm_sparse(bulk_counts_sparse, gene_lengths_aligned)
# Creamos el DataFrame final a partir de la matriz dispersa resultante
bulk_tpm_df = pd.DataFrame.sparse.from_spmatrix(
    bulk_tpm_sparse, index=bulk_counts_aligned.index, columns=bulk_counts_aligned.columns
)

In [ ]:
print("Normalizando datos de referencia a TPM...")
# Alinear y normalizar la referencia
ref_counts_aligned = adata_ref.raw.X[:, adata_ref.raw.var_names.isin(common_genes_with_length)]
# Obtenemos los nombres de los genes en el orden correcto
ref_gene_names_aligned = adata_ref.raw.var_names[adata_ref.raw.var_names.isin(common_genes_with_length)]
# Ordenamos las longitudes para que coincidan
gene_lengths_ref_aligned = gene_lengths[ref_gene_names_aligned]

# Aplicamos la nueva función directamente a la matriz dispersa
ref_tpm_sparse = counts_to_tpm_sparse(ref_counts_aligned.tocsr(), gene_lengths_ref_aligned)

In [ ]:
# 1. Convertimos la matriz dispersa de referencia a un DataFrame de pandas.
# Esto cargará los datos de TPM de la referencia en memoria.
print("Convirtiendo la matriz TPM de referencia a DataFrame de pandas...")
ref_tpm_df = pd.DataFrame.sparse.from_spmatrix(
    ref_tpm_sparse,
    index=adata_ref.obs.index,
    columns=ref_gene_names_aligned # Usamos la lista de nombres de genes ya alineada
)

ref_tpm_df['cell_type'] = adata_ref.obs['cell_type'].values

print("Agrupando por tipo celular y calculando la expresión promedio...")
signature_matrix_tpm = ref_tpm_df.groupby('cell_type').mean().T

print("Matriz de firmas creada con éxito.")
display(signature_matrix_tpm.head())

### 3.1.5. Alineamiento Final sobre Genes Marcadores

Filtramos tanto la matriz de firmas como la matriz de bulk para quedarnos únicamente con el subconjunto de genes marcadores que nuestro modelo de clasificación fue entrenado para reconocer.

In [ ]:
# Regenerar la lista de genes marcadores desde la referencia
if 'rank_genes_groups' not in adata_ref.uns:
    print("Calculando genes marcadores en la referencia...")
    sc.tl.rank_genes_groups(adata_ref, groupby='cell_type', method='t-test', use_raw=False)

marker_genes_df_from_ref = pd.DataFrame(adata_ref.uns['rank_genes_groups']['names'])
top_n_genes = 25 # Mismo valor que en el Notebook 2

# Extraer la lista de genes
marker_genes_list = []
for col in marker_genes_df_from_ref.columns:
    marker_genes_list.extend(marker_genes_df_from_ref[col].head(top_n_genes))

# Eliminamos duplicados y ordenamos
marker_genes_list = sorted(list(set(marker_genes_list)))
print(f"Se ha regenerado la lista de {len(marker_genes_list)} genes marcadores únicos.")

# Encontrar la intersección final de genes
common_markers = list(
    set(marker_genes_list) & 
    set(signature_matrix_tpm.index) & 
    set(bulk_tpm_df.columns)
)
print(f"Alineando sobre {len(common_markers)} genes marcadores comunes finales.")

# Filtrar las matrices finales
signature_matrix_aligned = signature_matrix_tpm.loc[common_markers]
bulk_tpm_aligned = bulk_tpm_df[common_markers] # Ahora es bulk_tpm_df, no bulk_counts_df

print("Datos de firma y de bulk armonizados a escala TPM y filtrados por marcadores.")

## 3.2.0 Prueba de Flujo Completo con un Subconjunto

Antes de lanzar la deconvolución completa (que es computacionalmente costosa), realizamos una prueba en un pequeño subconjunto de datos (~1%) para verificar que todo el pipeline de cálculo, guardado y re-lectura funciona correctamente.

In [ ]:
print("--- INICIANDO PRUEBA DE FLUJO COMPLETO CON SUBSET ---")

# 1. Crear un subconjunto pequeño de los datos de bulk
n_samples = len(bulk_tpm_aligned)
subset_size = int(n_samples * 0.02) # ~2% de las muestras
if subset_size < 5: subset_size = 5 # Aseguramos un mínimo de 5 muestras
bulk_subset = bulk_tpm_aligned.head(subset_size)
print(f"Subconjunto de prueba creado con {len(bulk_subset)} muestras.")

# 2. Ejecutar la deconvolución en el subconjunto
# (Este es el mismo código del bucle principal, pero sobre el subset)
X_signature_test = signature_matrix_aligned.values
cell_types_test = signature_matrix_aligned.columns
all_proportions_test = []

for sample_id in tqdm(bulk_subset.index, desc="Deconvolucionando subset"):
    y_bulk_sample_test = bulk_subset.loc[sample_id].values
    model_svr_test = NuSVR(kernel='linear', nu=0.5)
    model_svr_test.fit(X_signature_test, y_bulk_sample_test)
    raw_proportions_test = model_svr_test.coef_.copy()
    
    raw_proportions_test[raw_proportions_test < 0] = 0
    sum_proportions_test = raw_proportions_test.sum()
    if sum_proportions_test > 0:
        normalized_proportions_test = raw_proportions_test / sum_proportions_test
    else:
        normalized_proportions_test = raw_proportions_test
        
    all_proportions_test.append(normalized_proportions_test.flatten())

# 3. Crear el DataFrame de resultados del test
results_df_test = pd.DataFrame(
    all_proportions_test,
    index=bulk_subset.index,
    columns=cell_types_test
)

# 4. Aplicar la corrección de tipos y guardar
print("\nGuardando resultados del test...")
results_df_test.columns = results_df_test.columns.astype(str) # ¡Corrección clave!
test_output_path = os.path.join(DATA_PROCESSED_PATH, 'test_deconv_results.parquet')
results_df_test.to_parquet(test_output_path, engine='pyarrow')

# 5. Intentar leer el fichero guardado
print("Intentando leer los resultados del test guardados...")
try:
    loaded_test_df = pd.read_parquet(test_output_path)
    print("\n[ÉXITO] El fichero de prueba se ha guardado y leído correctamente.")
    print("Tipos de datos de las columnas leídas:")
    print(loaded_test_df.columns.dtype)
    display(loaded_test_df.head())
    
    # Limpiamos el fichero de prueba
    os.remove(test_output_path)
    print("\nFichero de prueba eliminado.")
    
except Exception as e:
    print(f"\n[FALLO] La prueba ha fallado. Error al leer el fichero: {e}")

## 3.2. Ejecución de la Deconvolución Basada en nu-SVR y guardado del archivo

Ahora, iteramos sobre cada una de las 530 muestras tumorales. Para cada una, ajustamos un modelo de regresión de soporte vectorial (nu-SVR) para estimar las proporciones de los 10 tipos celulares de nuestra firma.

In [ ]:
print("\n--- Iniciando el proceso de deconvolución ---")


DECONV_RESULTS_FILENAME = 'TCGA-LUAD_deconvolution_results.parquet'
deconv_output_path = os.path.join(DATA_PROCESSED_PATH, DECONV_RESULTS_FILENAME)

if os.path.exists(deconv_output_path):
    print(f"--- Encontrado fichero de resultados. Cargando desde: {deconv_output_path} ---")
    deconvolution_results_df = pd.read_parquet(deconv_output_path)
    cell_types = deconvolution_results_df.columns.tolist()

else :
    print(f"--- No se encontró fichero de resultados. Iniciando el proceso de deconvolución... ---")
    # Preparamos los datos para sklearn
    X_signature = signature_matrix_aligned.values
    cell_types = signature_matrix_aligned.columns

    # Lista para guardar los resultados
    all_proportions = []

    # Usamos tqdm para visualizar el progreso del bucle
    for sample_id in tqdm(bulk_tpm_aligned.index, desc="Deconvolucionando muestras"):
        # Obtenemos el perfil de expresión de la muestra actual
        y_bulk_sample = bulk_tpm_aligned.loc[sample_id].values
        
        # Definimos y entrenamos el modelo de regresión
        # nu=0.5 es un valor estándar. kernel='linear' porque asumimos una mezcla aditiva.
        model_svr = NuSVR(kernel='linear', nu=0.5)
        model_svr.fit(X_signature, y_bulk_sample)
        
        # Obtenemos los coeficientes del modelo. Estos son nuestras "proporciones" crudas.
        raw_proportions = model_svr.coef_.copy()
        
        # Post-procesamiento de los coeficientes:
        # 1. Forzar a que no sean negativos (biológicamente no tiene sentido una proporción negativa)
        raw_proportions[raw_proportions < 0] = 0
        
        # 2. Normalizar para que la suma de las proporciones sea 1 (si la suma no es cero)
        sum_proportions = raw_proportions.sum()
        if sum_proportions > 0:
            normalized_proportions = raw_proportions / sum_proportions
        else:
            normalized_proportions = raw_proportions # Se queda como un vector de ceros
            
        all_proportions.append(normalized_proportions.flatten())

    # Convertimos la lista de resultados en un DataFrame
    deconvolution_results_df = pd.DataFrame(
        all_proportions,
        index=bulk_tpm_aligned.index,
        columns=cell_types
    )

# Aseguramos que los tipos de datos de las columnas son simples antes de guardar.
deconvolution_results_df.columns = deconvolution_results_df.columns.astype(str)
# Guardamos los resultados para la próxima vez
deconvolution_results_df.to_parquet(deconv_output_path, engine='pyarrow')
print(f"\nResultados de la deconvolución calculados y guardados en: {deconv_output_path}")
print("\n--- Deconvolución completada ---")
print("Dimensiones del DataFrame de resultados:", deconvolution_results_df.shape)
display(deconvolution_results_df.head())


## 3.3. Verificación de los Resultados

Verificamos que las proporciones estimadas para cada muestra sumen 1.

In [ ]:
# Calculamos la suma de las proporciones para cada muestra
row_sums = deconvolution_results_df.sum(axis=1)

# Usamos np.allclose para verificar si todas las sumas son aproximadamente 1
try:
    assert np.allclose(row_sums, 1.0)
    print("\n[OK] Verificación exitosa: Todas las proporciones por muestra suman 1.")
except AssertionError:
    print("\n[ERROR] ¡Las proporciones no suman 1! Revisa el paso de normalización.")
    display(row_sums.describe())

# 4. Análisis Exploratorio de los Resultados de la Deconvolución

## 4.1. Fusión de Resultados con Datos Clínicos

El primer paso es unir nuestras proporciones celulares estimadas con la tabla de datos clínicos para facilitar los análisis posteriores.

In [ ]:
print("--- Fusionando resultados de deconvolución con datos clínicos ---")

# Nos aseguramos de que los índices coincidan antes de unir
try:
    assert all(deconvolution_results_df.index == bulk_clinical_df.index)
    # Usamos pd.concat para unir por columnas (axis=1)
    analysis_df = pd.concat([bulk_clinical_df, deconvolution_results_df], axis=1)
    print("Fusión completada con éxito.")
    print("Dimensiones del DataFrame de análisis final:", analysis_df.shape)
    display(analysis_df.head())
except AssertionError:
    print("[ERROR] Los índices entre los resultados de deconvolución y los datos clínicos no coinciden.")

## 4.2. Composición Celular Promedio de la Cohorte

Para obtener una visión general, calculamos y visualizamos la proporción promedio de cada tipo celular en toda la cohorte de tumores.

In [ ]:
print("\n--- Visualizando la composición celular promedio ---")

# Calculamos la media de cada columna de tipo celular
mean_proportions = analysis_df[cell_types].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=mean_proportions.index, y=mean_proportions.values)
plt.title('Composición Celular Promedio en la Cohorte TCGA-LUAD', fontsize=16)
plt.ylabel('Proporción Promedio')
plt.xlabel('Tipo Celular')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(mean_proportions.to_frame(name='Proporción Promedio'))

### 4.2.1. Validación Cruzada con Proporciones de Referencia scRNA-seq

Para validar la plausibilidad de nuestras estimaciones de deconvolución, comparamos la composición celular promedio obtenida en la cohorte de TCGA con las proporciones celulares reales observadas en nuestro dataset de referencia de scRNA-seq. Aunque no se espera una coincidencia perfecta debido a diferencias tecnológicas y de cohorte, una correlación general en el ranking de abundancia de los tipos celulares aumentaría la confianza en nuestro método.


In [ ]:
print("\n--- Comparando las proporciones de deconvolución con las de la referencia scRNA-seq ---")

# 1. Calcular las proporciones reales en el dataset de scRNA-seq de referencia
# Usamos el objeto `adata_ref` que cargamos al principio
sc_proportions = adata_ref.obs['cell_type'].value_counts(normalize=True).sort_values(ascending=False)

# 2. Obtener las proporciones promedio de la deconvolución (ya las teníamos)
deconv_proportions = mean_proportions # 'mean_proportions' de la celda anterior

# 3. Crear un DataFrame combinado para facilitar la visualización
comparison_df = pd.DataFrame({
    'Deconvolución (TCGA)': deconv_proportions,
    'Referencia (scRNA-seq)': sc_proportions
}).fillna(0) # Rellenamos con 0 si algún tipo celular no estuviera en uno de los sets

# 4. Visualizar la comparación
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Comparación de Proporciones Celulares: Deconvolución vs. Referencia scRNA-seq', fontsize=16)

# Gráfico de barras agrupado
comparison_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Composición Promedio por Método')
axes[0].set_ylabel('Proporción')
axes[0].set_xlabel('Tipo Celular')
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot para ver la correlación
sns.regplot(data=comparison_df, x='Referencia (scRNA-seq)', y='Deconvolución (TCGA)', ax=axes[1])
axes[1].set_title('Correlación entre Proporciones Estimadas y Reales')
axes[1].set_xlabel('Proporción en scRNA-seq (Real)')
axes[1].set_ylabel('Proporción en Deconvolución (Estimada)')
# Añadir línea de identidad (y=x) para una comparación perfecta
max_val = comparison_df.max().max()
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Identidad (y=x)')
axes[1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

display(comparison_df)